In [1]:
from pyspark.sql.functions import current_timestamp, input_file_name
from pyspark.sql.types import *

# مسار الملفات الخام
raw_path = "Files/raw/"

# اسم كل ملف CSV -> اسم الجدول اللي هيتحط في ODS
files_map = {
    "olist_customers_dataset.csv": "ods_customers",
    "olist_geolocation_dataset.csv": "ods_geolocation",
    "olist_order_items_dataset.csv": "ods_order_items",
    "olist_order_payments_dataset.csv": "ods_order_payments",
    "olist_order_reviews_dataset.csv": "ods_order_reviews",
    "olist_orders_dataset.csv": "ods_orders",
    "olist_products_dataset.csv": "ods_products",
    "olist_sellers_dataset.csv": "ods_sellers",
    "product_category_name_translation.csv": "ods_product_category_translation"
}

StatementMeta(, 0f8b934e-8fb8-4b58-9251-cb3249ee67a2, 3, Finished, Available, Finished, False)

In [2]:
for file_name, table_name in files_map.items():
    file_path = raw_path + file_name
    
    print(f"⏳ Reading {file_name} ...")
    
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .option("multiLine", "true")     # مهم لملف الـ reviews (فيه نصوص متعددة الأسطر)
          .option("escape", "\"")
          .csv(file_path))
    
    # أعمدة رقابة (Audit Columns) — أساسية في أي ODS Layer
    df = (df
          .withColumn("_ingestion_timestamp", current_timestamp())
          .withColumn("_source_file", input_file_name()))
    
    # الكتابة كـ Delta Table في schema الـ ods
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"ods.{table_name}"))
    
    print(f" Loaded into ods.{table_name} | Rows: {df.count()} | Cols: {len(df.columns)}")

print("\n Successfully loaded all files into ODS schema")

StatementMeta(, 0f8b934e-8fb8-4b58-9251-cb3249ee67a2, 4, Finished, Available, Finished, False)

⏳ Reading olist_customers_dataset.csv ...
 Loaded into ods.ods_customers | Rows: 99441 | Cols: 7
⏳ Reading olist_geolocation_dataset.csv ...
 Loaded into ods.ods_geolocation | Rows: 1000163 | Cols: 7
⏳ Reading olist_order_items_dataset.csv ...
 Loaded into ods.ods_order_items | Rows: 112650 | Cols: 9
⏳ Reading olist_order_payments_dataset.csv ...
 Loaded into ods.ods_order_payments | Rows: 103886 | Cols: 7
⏳ Reading olist_order_reviews_dataset.csv ...
 Loaded into ods.ods_order_reviews | Rows: 99224 | Cols: 9
⏳ Reading olist_orders_dataset.csv ...
 Loaded into ods.ods_orders | Rows: 99441 | Cols: 10
⏳ Reading olist_products_dataset.csv ...
 Loaded into ods.ods_products | Rows: 32951 | Cols: 11
⏳ Reading olist_sellers_dataset.csv ...
 Loaded into ods.ods_sellers | Rows: 3095 | Cols: 6
⏳ Reading product_category_name_translation.csv ...
 Loaded into ods.ods_product_category_translation | Rows: 71 | Cols: 4

 Successfully loaded all files into ODS schema


In [3]:
tables = spark.sql("SHOW TABLES IN ods")
display(tables)

StatementMeta(, 0f8b934e-8fb8-4b58-9251-cb3249ee67a2, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c23cf997-e37f-4362-bd48-c43d09686389)